In [2]:
import pandas as pd
import numpy as np
import joblib
import pickle

from xgboost import XGBRanker
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import spearmanr

df = pd.read_csv("f1_ml_ready_dataset.csv", low_memory=False)

print("Dataset loaded.")
print("Original shape:", df.shape)

required_cols = ["raceId", "driverId", "positionOrder"]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["positionOrder"] = pd.to_numeric(df["positionOrder"], errors="coerce")
df = df.dropna(subset=["positionOrder"]).copy()
df["positionOrder"] = df["positionOrder"].astype(int)
df = df[df["positionOrder"] > 0].copy()

df = df.sort_values(["raceId", "driverId"]).drop_duplicates(
    subset=["raceId", "driverId"],
    keep="last"
).copy()

df["race_group"] = df["raceId"].astype(str)

group_sizes = df.groupby("race_group").size()
valid_groups = group_sizes[group_sizes >= 2].index
df = df[df["race_group"].isin(valid_groups)].copy()

print("Shape after dedupe/filter:", df.shape)

df["max_pos_in_race"] = df.groupby("race_group")["positionOrder"].transform("max")
df["relevance"] = df["max_pos_in_race"] + 1 - df["positionOrder"]

df["relevance"] = pd.to_numeric(df["relevance"], errors="coerce")
df = df.dropna(subset=["relevance"]).copy()
df["relevance"] = df["relevance"].astype(np.int32)

df = df[df["relevance"] >= 0].copy()

print("Relevance min:", df["relevance"].min())
print("Relevance max:", df["relevance"].max())

drop_cols = [
    "position",
    "positionOrder",
    "race_points",
    "is_podium",
    "is_win",
    "positions_gained",
    "laps",
    "status",
    "statusId",
    "fastestLap",
    "relevance",
    "max_pos_in_race",
    "race_group"
]

drop_cols = [c for c in drop_cols if c in df.columns]

feature_columns = [c for c in df.columns if c not in drop_cols]

print("Feature count:", len(feature_columns))

X = df[feature_columns].copy()
y = df["relevance"].copy()
groups = df["race_group"].copy()

label_encoders = {}

for col in X.columns:
    if X[col].dtype == "object":
        X[col] = X[col].astype(str).fillna("UNKNOWN")

        unique_vals = sorted(X[col].unique())
        mapping = {v: i for i, v in enumerate(unique_vals)}

        X[col] = X[col].map(mapping).astype(np.int32)
        label_encoders[col] = mapping

    else:
        X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0)

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0)

X = X.astype(np.float32)
y = y.astype(np.int32)

print("Object columns after encoding:", X.select_dtypes(include=["object"]).columns.tolist())
print("Null count in X:", X.isnull().sum().sum())
print("Null count in y:", y.isnull().sum())

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

meta_train = df.iloc[train_idx].copy()
meta_test = df.iloc[test_idx].copy()

train_df = X_train.copy()
train_df["target"] = y_train.values
train_df["qid"] = groups_train.values

test_df = X_test.copy()
test_df["target"] = y_test.values
test_df["qid"] = groups_test.values

meta_train["qid"] = groups_train.values
meta_test["qid"] = groups_test.values

train_df = train_df.sort_values("qid").reset_index(drop=True)
test_df = test_df.sort_values("qid").reset_index(drop=True)

meta_train = meta_train.sort_values("qid").reset_index(drop=True)
meta_test = meta_test.sort_values("qid").reset_index(drop=True)

X_train_sorted = train_df.drop(columns=["target", "qid"]).astype(np.float32)
y_train_sorted = train_df["target"].astype(np.int32)

X_test_sorted = test_df.drop(columns=["target", "qid"]).astype(np.float32)
y_test_sorted = test_df["target"].astype(np.int32)

train_qid = pd.factorize(train_df["qid"])[0].astype(np.uint32)
test_qid = pd.factorize(test_df["qid"])[0].astype(np.uint32)

print("Train rows:", len(X_train_sorted))
print("Test rows:", len(X_test_sorted))
print("Train qid unique:", len(np.unique(train_qid)))
print("Test qid unique:", len(np.unique(test_qid)))

ranker = XGBRanker(
    objective="rank:pairwise",
    eval_metric="ndcg",
    learning_rate=0.05,
    max_depth=6,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

ranker.fit(
    X_train_sorted,
    y_train_sorted,
    qid=train_qid,
    verbose=True
)

print("Model trained successfully.")

meta_test = meta_test.reset_index(drop=True).copy()
meta_test["ranking_score"] = ranker.predict(X_test_sorted)

race_scores = []

for race_id, group_df in meta_test.groupby("qid"):
    group_df = group_df.copy()

    actual_sorted = group_df.sort_values("positionOrder")
    predicted_sorted = group_df.sort_values("ranking_score", ascending=False)

    actual_order = actual_sorted["driverId"].tolist()
    predicted_order = predicted_sorted["driverId"].tolist()

    actual_rank_map = {d: i + 1 for i, d in enumerate(actual_order)}
    predicted_rank_map = {d: i + 1 for i, d in enumerate(predicted_order)}

    common = [d for d in actual_rank_map if d in predicted_rank_map]

    actual_ranks = [actual_rank_map[d] for d in common]
    predicted_ranks = [predicted_rank_map[d] for d in common]

    if len(actual_ranks) > 1:
        corr, _ = spearmanr(actual_ranks, predicted_ranks)
        race_scores.append(corr)

mean_spearman = np.nanmean(race_scores)

print("Mean Spearman Rank Correlation:", round(float(mean_spearman), 4))

rank_results = []

for race_id, group_df in meta_test.groupby("qid"):
    group_df = group_df.copy()

    group_df["actual_rank"] = group_df["positionOrder"].rank(
        method="first",
        ascending=True
    ).astype(int)

    group_df["predicted_rank"] = group_df["ranking_score"].rank(
        method="first",
        ascending=False
    ).astype(int)

    group_df["rank_difference"] = group_df["predicted_rank"] - group_df["actual_rank"]

    rank_results.append(group_df)

rank_comparison_df = pd.concat(rank_results, ignore_index=True)

display_cols = []

possible_display_cols = [
    "raceId",
    "year",
    "round",
    "name",
    "raceName",
    "driverId",
    "driverRef",
    "driver_name",
    "forename",
    "surname",
    "constructorId",
    "constructorRef",
    "constructor_name",
    "positionOrder",
    "actual_rank",
    "ranking_score",
    "predicted_rank",
    "rank_difference"
]

for col in possible_display_cols:
    if col in rank_comparison_df.columns:
        display_cols.append(col)

rank_comparison_table = rank_comparison_df[display_cols].copy()

sort_cols = []

if "raceId" in rank_comparison_table.columns:
    sort_cols.append("raceId")

sort_cols.append("predicted_rank")

rank_comparison_table = rank_comparison_table.sort_values(sort_cols).reset_index(drop=True)

print("\nActual Rank vs Predicted Rank Table:")
print(rank_comparison_table.head(50))

try:
    from IPython.display import display
    display(rank_comparison_table.head(50))
except:
    pass

rank_comparison_table.to_csv("actual_vs_predicted_rank_table.csv", index=False)

print("\nActual vs predicted rank table saved as:")
print("- actual_vs_predicted_rank_table.csv")

with open("f1_xgb_ranker.pkl", "wb") as f:
    pickle.dump(ranker, f)

joblib.dump(label_encoders, "ranker_label_encoders.pkl")
joblib.dump(feature_columns, "ranker_feature_columns.pkl")

print("\nSaved:")
print("- f1_xgb_ranker.pkl")
print("- ranker_label_encoders.pkl")
print("- ranker_feature_columns.pkl")
print("- actual_vs_predicted_rank_table.csv")

Dataset loaded.
Original shape: (26759, 53)
Shape after dedupe/filter: (26668, 54)
Relevance min: 1
Relevance max: 39
Feature count: 52
Object columns after encoding: []
Null count in X: 0
Null count in y: 0
Train rows: 21370
Test rows: 5298
Train qid unique: 900
Test qid unique: 225
Model trained successfully.
Mean Spearman Rank Correlation: 0.698

Actual Rank vs Predicted Rank Table:
    raceId  year  round  driverId  constructorId  constructor_name  \
0        5  2009      5        18             23                31   
1        5  2009      5        20              9               166   
2        5  2009      5        22             23                31   
3        5  2009      5        13              6                73   
4        5  2009      5        17              9               166   
5        5  2009      5         3              3               208   
6        5  2009      5         4              4               167   
7        5  2009      5         2              2   

,raceId,year,round,driverId,constructorId,constructor_name,positionOrder,actual_rank,ranking_score,predicted_rank,rank_difference
0,5,2009,5,18,23,31,1,1,2.267861,1,0
1,5,2009,5,20,9,166,4,4,1.837455,2,-2
2,5,2009,5,22,23,31,2,2,1.630311,3,1
3,5,2009,5,13,6,73,6,6,1.481314,4,-2
4,5,2009,5,17,9,166,3,3,1.200492,5,2
5,5,2009,5,3,3,208,8,8,0.958477,6,-2
6,5,2009,5,4,4,167,5,5,0.938410,7,2
7,5,2009,5,2,2,17,7,7,0.468977,8,1
8,5,2009,5,10,7,198,10,10,-0.292334,9,-1
9,5,2009,5,12,4,167,12,12,-0.360684,10,-2



Actual vs predicted rank table saved as:
- actual_vs_predicted_rank_table.csv

Saved:
- f1_xgb_ranker.pkl
- ranker_label_encoders.pkl
- ranker_feature_columns.pkl
- actual_vs_predicted_rank_table.csv


In [3]:
import pandas as pd
import numpy as np
import joblib
import pickle

from xgboost import XGBRanker
from sklearn.model_selection import GroupShuffleSplit, ParameterSampler
from scipy.stats import spearmanr

df = pd.read_csv("f1_ml_ready_dataset.csv", low_memory=False)

print("Dataset loaded.")
print("Original shape:", df.shape)

required_cols = ["raceId", "driverId", "positionOrder"]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["positionOrder"] = pd.to_numeric(df["positionOrder"], errors="coerce")
df = df.dropna(subset=["positionOrder"]).copy()
df["positionOrder"] = df["positionOrder"].astype(int)
df = df[df["positionOrder"] > 0].copy()

df = df.sort_values(["raceId", "driverId"]).drop_duplicates(
    subset=["raceId", "driverId"],
    keep="last"
).copy()

df["race_group"] = df["raceId"].astype(str)

group_sizes = df.groupby("race_group").size()
valid_groups = group_sizes[group_sizes >= 2].index
df = df[df["race_group"].isin(valid_groups)].copy()

print("Shape after dedupe/filter:", df.shape)

df["max_pos_in_race"] = df.groupby("race_group")["positionOrder"].transform("max")
df["relevance"] = df["max_pos_in_race"] + 1 - df["positionOrder"]

df["relevance"] = pd.to_numeric(df["relevance"], errors="coerce")
df = df.dropna(subset=["relevance"]).copy()
df["relevance"] = df["relevance"].astype(np.int32)

df = df[df["relevance"] >= 0].copy()

print("Relevance min:", df["relevance"].min())
print("Relevance max:", df["relevance"].max())

drop_cols = [
    "position",
    "positionOrder",
    "race_points",
    "is_podium",
    "is_win",
    "positions_gained",
    "laps",
    "status",
    "statusId",
    "fastestLap",
    "relevance",
    "max_pos_in_race",
    "race_group"
]

drop_cols = [c for c in drop_cols if c in df.columns]

feature_columns = [c for c in df.columns if c not in drop_cols]

print("Feature count:", len(feature_columns))

X = df[feature_columns].copy()
y = df["relevance"].copy()
groups = df["race_group"].copy()

label_encoders = {}

for col in X.columns:
    if X[col].dtype == "object":
        X[col] = X[col].astype(str).fillna("UNKNOWN")

        unique_vals = sorted(X[col].unique())
        mapping = {v: i for i, v in enumerate(unique_vals)}

        X[col] = X[col].map(mapping).astype(np.int32)
        label_encoders[col] = mapping
    else:
        X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0)

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0)

X = X.astype(np.float32)
y = y.astype(np.int32)

print("Object columns after encoding:", X.select_dtypes(include=["object"]).columns.tolist())
print("Null count in X:", X.isnull().sum().sum())
print("Null count in y:", y.isnull().sum())

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_full = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train_full = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train_full = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

meta_train_full = df.iloc[train_idx].copy()
meta_test = df.iloc[test_idx].copy()

print("Full train rows:", len(X_train_full))
print("Final test rows:", len(X_test))
print("Full train races:", groups_train_full.nunique())
print("Final test races:", groups_test.nunique())

val_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

inner_train_idx, val_idx = next(
    val_splitter.split(X_train_full, y_train_full, groups=groups_train_full)
)

X_inner_train = X_train_full.iloc[inner_train_idx].copy()
X_val = X_train_full.iloc[val_idx].copy()

y_inner_train = y_train_full.iloc[inner_train_idx].copy()
y_val = y_train_full.iloc[val_idx].copy()

groups_inner_train = groups_train_full.iloc[inner_train_idx].copy()
groups_val = groups_train_full.iloc[val_idx].copy()

meta_inner_train = meta_train_full.iloc[inner_train_idx].copy()
meta_val = meta_train_full.iloc[val_idx].copy()

print("Inner train rows:", len(X_inner_train))
print("Validation rows:", len(X_val))
print("Inner train races:", groups_inner_train.nunique())
print("Validation races:", groups_val.nunique())

def prepare_ranker_data(X_data, y_data, groups_data, meta_data):
    temp_df = X_data.copy()
    temp_df["target"] = y_data.values
    temp_df["qid"] = groups_data.values

    meta_data = meta_data.copy()
    meta_data["qid"] = groups_data.values

    temp_df = temp_df.sort_values("qid").reset_index(drop=True)
    meta_data = meta_data.sort_values("qid").reset_index(drop=True)

    X_sorted = temp_df.drop(columns=["target", "qid"]).astype(np.float32)
    y_sorted = temp_df["target"].astype(np.int32)

    qid = pd.factorize(temp_df["qid"])[0].astype(np.uint32)

    return X_sorted, y_sorted, qid, meta_data


X_inner_train_sorted, y_inner_train_sorted, train_qid, meta_inner_train_sorted = prepare_ranker_data(
    X_inner_train,
    y_inner_train,
    groups_inner_train,
    meta_inner_train
)

X_val_sorted, y_val_sorted, val_qid, meta_val_sorted = prepare_ranker_data(
    X_val,
    y_val,
    groups_val,
    meta_val
)

X_test_sorted, y_test_sorted, test_qid, meta_test_sorted = prepare_ranker_data(
    X_test,
    y_test,
    groups_test,
    meta_test
)

print("Prepared ranker data successfully.")

def evaluate_spearman(model, X_eval, meta_eval):
    meta_eval = meta_eval.reset_index(drop=True).copy()
    meta_eval["ranking_score"] = model.predict(X_eval)

    race_scores = []

    for race_id, group_df in meta_eval.groupby("qid"):
        group_df = group_df.copy()

        actual_sorted = group_df.sort_values("positionOrder")
        predicted_sorted = group_df.sort_values("ranking_score", ascending=False)

        actual_order = actual_sorted["driverId"].tolist()
        predicted_order = predicted_sorted["driverId"].tolist()

        actual_rank_map = {d: i + 1 for i, d in enumerate(actual_order)}
        predicted_rank_map = {d: i + 1 for i, d in enumerate(predicted_order)}

        common_drivers = [d for d in actual_rank_map if d in predicted_rank_map]

        actual_ranks = [actual_rank_map[d] for d in common_drivers]
        predicted_ranks = [predicted_rank_map[d] for d in common_drivers]

        if len(actual_ranks) > 1:
            corr, _ = spearmanr(actual_ranks, predicted_ranks)

            if not np.isnan(corr):
                race_scores.append(corr)

    if len(race_scores) == 0:
        return np.nan

    return float(np.mean(race_scores))

param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "max_depth": [3, 4, 5, 6, 8],
    "min_child_weight": [1, 3, 5, 7],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.3, 0.5, 1],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [0.5, 1, 1.5, 2, 3]
}

n_iter = 30

param_list = list(
    ParameterSampler(
        param_grid,
        n_iter=n_iter,
        random_state=42
    )
)

print(f"Total hyperparameter combinations to test: {len(param_list)}")

best_score = -999
best_params = None
best_model = None

tuning_results = []

for i, params in enumerate(param_list, start=1):
    print("\n==============================")
    print(f"Training model {i}/{len(param_list)}")
    print("Params:", params)

    model = XGBRanker(
        objective="rank:pairwise",
        eval_metric="ndcg",
        random_state=42,
        tree_method="hist",
        **params
    )

    model.fit(
        X_inner_train_sorted,
        y_inner_train_sorted,
        qid=train_qid,
        verbose=False
    )

    val_score = evaluate_spearman(
        model,
        X_val_sorted,
        meta_val_sorted
    )

    print("Validation Spearman:", round(val_score, 4))

    tuning_results.append({
        "iteration": i,
        "spearman_score": val_score,
        **params
    })

    if val_score > best_score:
        best_score = val_score
        best_params = params
        best_model = model

        print("New best model found.")

print("\n==============================")
print("Hyperparameter tuning completed.")
print("Best Validation Spearman:", round(best_score, 4))
print("Best Parameters:")
print(best_params)

tuning_results_df = pd.DataFrame(tuning_results)
tuning_results_df = tuning_results_df.sort_values(
    "spearman_score",
    ascending=False
)

tuning_results_df.to_csv("xgb_ranker_tuning_results.csv", index=False)

print("Saved tuning results to xgb_ranker_tuning_results.csv")

X_train_full_sorted, y_train_full_sorted, full_train_qid, meta_train_full_sorted = prepare_ranker_data(
    X_train_full,
    y_train_full,
    groups_train_full,
    meta_train_full
)

final_ranker = XGBRanker(
    objective="rank:pairwise",
    eval_metric="ndcg",
    random_state=42,
    tree_method="hist",
    **best_params
)

final_ranker.fit(
    X_train_full_sorted,
    y_train_full_sorted,
    qid=full_train_qid,
    verbose=True
)

print("Final model trained using best hyperparameters.")

test_spearman = evaluate_spearman(
    final_ranker,
    X_test_sorted,
    meta_test_sorted
)

print("Final Test Spearman Rank Correlation:", round(test_spearman, 4))

with open("f1_xgb_ranker_tuned.pkl", "wb") as f:
    pickle.dump(final_ranker, f)

joblib.dump(label_encoders, "ranker_label_encoders.pkl")
joblib.dump(feature_columns, "ranker_feature_columns.pkl")
joblib.dump(best_params, "xgb_ranker_best_params.pkl")

print("\nSaved:")
print("- f1_xgb_ranker_tuned.pkl")
print("- ranker_label_encoders.pkl")
print("- ranker_feature_columns.pkl")
print("- xgb_ranker_best_params.pkl")
print("- xgb_ranker_tuning_results.csv")

Dataset loaded.
Original shape: (26759, 53)
Shape after dedupe/filter: (26668, 54)
Relevance min: 1
Relevance max: 39
Feature count: 52
Object columns after encoding: []
Null count in X: 0
Null count in y: 0
Full train rows: 21370
Final test rows: 5298
Full train races: 900
Final test races: 225
Inner train rows: 17033
Validation rows: 4337
Inner train races: 720
Validation races: 180
Prepared ranker data successfully.
Total hyperparameter combinations to test: 30

Training model 1/30
Params: {'subsample': 0.9, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.01, 'gamma': 0.5, 'colsample_bytree': 0.6}
Validation Spearman: 0.7514
New best model found.

Training model 2/30
Params: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 500, 'min_child_weight': 3, 'max_depth': 8, 'learning_rate': 0.08, 'gamma': 0.1, 'colsample_bytree': 0.9}
Validation Spearman: 0.7274

Training model 3/30
Params: {'subsample': 

In [4]:
pip install fastapi uvicorn pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!uvicorn main:app --reload --host 0.0.0.0 --port 8000

INFO:     Will watch for changes in these directories: ['/Users/pyaesone/Formula One Model Training']
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [45231] using StatReload
Feature engineered dataset columns:
['raceId', 'driverId', 'constructorId', 'number_x', 'grid', 'positionOrder', 'milliseconds', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'number_y', 'driver_nationality', 'constructor_name', 'constructor_nationality', 'year', 'round', 'circuitId', 'race_name', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time', 'driver_age', 'finished_race', 'driver_experience', 'constructor_experience', 'driver_circuit_experience', 'driver_avg_finish_before_race', 'driver_avg_points_before_race', 'driver_points_last_3', 'driver_points_last_5', 'driver_finish_last_3', 'driver_finish_last_5', 'previous_race_finish', 'previous_race_points', 'previous_grid